# Weather data analysis

This notebook loads the wide JSON export and plots daily outdoor temperature extremes, average relative humidity, and rainfall. The JSON currently contains 5-minute readings; daily summaries group readings by calendar day in the timestamps' local time.

If you move or rename the data file, update `DATA_PATH` in the next cell. Dependencies are managed by `uv` from `pyproject.toml`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_PATH = Path("data/easyweather/easyweather_wide.json")
required_columns = {
    "timestamp",
    "outdoor_temperature",
    "outdoor_humidity",
    "rainfall_piezo_daily",
}

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Set DATA_PATH to a wide JSON file."
    )

weather = pd.read_json(DATA_PATH, convert_dates=False)
missing_columns = required_columns.difference(weather.columns)
if missing_columns:
    raise ValueError(f"Missing expected columns: {sorted(missing_columns)}")

weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="raise")
weather["outdoor_temperature"] = pd.to_numeric(
    weather["outdoor_temperature"], errors="coerce"
)

daily_temperature = (
    weather.dropna(subset=["outdoor_temperature"])
    .set_index("timestamp")["outdoor_temperature"]
    .resample("D")
    .agg(min_temperature="min", max_temperature="max")
    .dropna(how="all")
)

print(f"Loaded {len(weather):,} readings from {DATA_PATH}")
print(f"Daily groups: {len(daily_temperature)}")
daily_temperature

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    daily_temperature.index,
    daily_temperature["max_temperature"],
    marker="o",
    label="Daily maximum",
    color="firebrick",
)
ax.plot(
    daily_temperature.index,
    daily_temperature["min_temperature"],
    marker="o",
    label="Daily minimum",
    color="royalblue",
)
ax.fill_between(
    daily_temperature.index,
    daily_temperature["min_temperature"],
    daily_temperature["max_temperature"],
    color="slategray",
    alpha=0.15,
    label="Daily range",
)

ax.set_title("Outdoor temperature: daily minimum and maximum")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.grid(True, alpha=0.3)
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Daily relative humidity and rainfall

This chart compares daily average outdoor relative humidity with daily rainfall totals. Daily rainfall is taken as the maximum of the cumulative `rainfall_piezo_daily` readings for each day. The two measurements use separate y-axes.

In [ ]:
weather["outdoor_humidity"] = pd.to_numeric(
    weather["outdoor_humidity"], errors="coerce"
)
weather["rainfall_piezo_daily"] = pd.to_numeric(
    weather["rainfall_piezo_daily"], errors="coerce"
)

daily_humidity_rain = (
    weather.set_index("timestamp")
    .resample("D")
    .agg(
        mean_relative_humidity=("outdoor_humidity", "mean"),
        daily_rainfall=("rainfall_piezo_daily", "max"),
    )
    .dropna(how="all")
)

fig, humidity_ax = plt.subplots(figsize=(12, 5))
rain_ax = humidity_ax.twinx()

humidity_ax.plot(
    daily_humidity_rain.index,
    daily_humidity_rain["mean_relative_humidity"],
    marker="o",
    color="darkgreen",
    label="Mean relative humidity",
)
rain_ax.bar(
    daily_humidity_rain.index,
    daily_humidity_rain["daily_rainfall"],
    width=0.8,
    color="steelblue",
    alpha=0.35,
    label="Daily rainfall",
)

humidity_ax.set_title("Daily relative humidity and rainfall")
humidity_ax.set_xlabel("Date")
humidity_ax.set_ylabel("Relative humidity (%)", color="darkgreen")
humidity_ax.set_ylim(0, 100)
rain_ax.set_ylabel("Rainfall (mm)", color="steelblue")
rain_ax.set_ylim(bottom=0)
humidity_ax.grid(True, alpha=0.3)
handles_1, labels_1 = humidity_ax.get_legend_handles_labels()
handles_2, labels_2 = rain_ax.get_legend_handles_labels()
humidity_ax.legend(handles_1 + handles_2, labels_1 + labels_2)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()

## Is rainfall higher when air is closer to the dew point?

The dew-point spread is outdoor temperature minus outdoor dew point, in °C. A smaller spread means the air is closer to saturation. Each point represents one day: the x-axis is that day's average spread and the y-axis is its rainfall total. Spearman's rank correlation summarizes the association; it does not show that a small spread causes rain.

In [ ]:
dewpoint_rain_columns = {
    "outdoor_temperature",
    "outdoor_dew_point",
    "rainfall_piezo_daily",
}
missing_dewpoint_rain_columns = dewpoint_rain_columns.difference(
    weather.columns
)
if missing_dewpoint_rain_columns:
    raise ValueError(
        f"Missing expected columns: {sorted(missing_dewpoint_rain_columns)}"
    )

weather["outdoor_dew_point"] = pd.to_numeric(
    weather["outdoor_dew_point"], errors="coerce"
)
weather["rainfall_piezo_daily"] = pd.to_numeric(
    weather["rainfall_piezo_daily"], errors="coerce"
)
weather["dew_point_spread"] = (
    weather["outdoor_temperature"] - weather["outdoor_dew_point"]
)

daily_dewpoint_rain = (
    weather.set_index("timestamp")
    .resample("D")
    .agg(
        mean_dew_point_spread=("dew_point_spread", "mean"),
        daily_rainfall=("rainfall_piezo_daily", "max"),
    )
)
paired_days = daily_dewpoint_rain.dropna(
    subset=["mean_dew_point_spread", "daily_rainfall"]
)
if paired_days.empty:
    raise ValueError("No days have both dew-point spread and rainfall data.")

spearman_rho = paired_days["mean_dew_point_spread"].rank().corr(
    paired_days["daily_rainfall"].rank()
)
dry_days = paired_days[paired_days["daily_rainfall"] == 0]
rainy_days = paired_days[paired_days["daily_rainfall"] > 0]

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(
    dry_days["mean_dew_point_spread"],
    dry_days["daily_rainfall"],
    color="gray",
    alpha=0.7,
    label="No rain",
)
ax.scatter(
    rainy_days["mean_dew_point_spread"],
    rainy_days["daily_rainfall"],
    color="steelblue",
    alpha=0.75,
    label="Rain",
)
ax.set_title("Daily rainfall vs. distance from dew point")
ax.set_xlabel("Daily mean outdoor temperature − dew point (°C)")
ax.set_ylabel("Daily rainfall (mm)")
ax.grid(True, alpha=0.3)
ax.legend()
correlation_text = (
    f"Spearman ρ = {spearman_rho:.2f} (n={len(paired_days)})"
    if pd.notna(spearman_rho)
    else f"Spearman ρ not defined (n={len(paired_days)})"
)
ax.text(
    0.02,
    0.98,
    correlation_text,
    transform=ax.transAxes,
    va="top",
    bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "none"},
)
fig.tight_layout()
plt.show()

### Daily dew-point spread and rainfall over time

This time-series view shows whether days with more rain also have a smaller average gap between outdoor temperature and dew point. The gap and rainfall use separate y-axes.

In [ ]:
fig, spread_ax = plt.subplots(figsize=(12, 5))
rain_ax = spread_ax.twinx()

spread_ax.plot(
    daily_dewpoint_rain.index,
    daily_dewpoint_rain["mean_dew_point_spread"],
    marker="o",
    markersize=3,
    color="darkorange",
    label="Mean temperature − dew point",
)
rain_ax.bar(
    daily_dewpoint_rain.index,
    daily_dewpoint_rain["daily_rainfall"],
    width=0.8,
    color="steelblue",
    alpha=0.35,
    label="Daily rainfall",
)

spread_ax.set_title("Daily dew-point spread and rainfall")
spread_ax.set_xlabel("Date")
spread_ax.set_ylabel("Temperature − dew point (°C)", color="darkorange")
spread_ax.set_ylim(bottom=0)
rain_ax.set_ylabel("Daily rainfall (mm)", color="steelblue")
rain_ax.set_ylim(bottom=0)
spread_ax.grid(True, alpha=0.3)
handles_1, labels_1 = spread_ax.get_legend_handles_labels()
handles_2, labels_2 = rain_ax.get_legend_handles_labels()
spread_ax.legend(handles_1 + handles_2, labels_1 + labels_2)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()